# ACI-DQN 数据中心调度 —— 云端训练脚本

本 Notebook 用于在云端（Colab / Kaggle / AutoDL 等）训练 RL 调度策略。

## 方法总览

| 类型 | 方法 | 状态维度 | 说明 |
|------|------|----------|------|
| RL | DQN | 17 | 标准 DQN，无不确定性特征 |
| RL | Forecast-DQN | 20 | DQN + K 维滚动均值点预测特征 |
| RL | Static-Conformal-DQN | 23 | DQN + 2K 维固定共形区间特征 |
| **RL** | **ACI-DQN** | **23** | **本文方法**：DQN + 2K 维 ACI 在线自适应区间 |

训练完成后，模型保存到 `outputs/models/`，可直接下载用于本地评估。

## 1. 环境配置

In [ ]:
# 安装依赖（云端环境按需取消注释）
%pip install numpy pandas matplotlib torch pyyaml scipy

In [ ]:
# ============================================================
# 检测项目根目录（适配 Colab / Kaggle / AutoDL 等不同环境）
# ============================================================
import os
from pathlib import Path

def _find_project_root():
    """向上搜索包含 src/utils.py 的目录作为项目根目录。"""
    # 候选起点
    candidates = [Path.cwd()]
    # 常见云端挂载路径
    for p in ["/content", "/content/drive/MyDrive/ACI-DQN",
              "/kaggle/working", "/root/autodl-tmp"]:
        if os.path.isdir(p):
            candidates.insert(0, Path(p))

    seen = set()
    for start in candidates:
        d = start.resolve()
        for _ in range(6):  # 最多向上 6 层
            if d in seen:
                break
            seen.add(d)
            if (d / "src" / "utils.py").exists() and (d / "config.yaml").exists():
                return d
            d = d.parent
    return None

PROJECT_ROOT = _find_project_root()

if PROJECT_ROOT is None:
    # 在 Colab 上，可能需要先上传项目文件
    import sys
    print("=" * 60)
    print("未找到项目文件！请先上传整个项目目录。")
    print("")
    print("方式 1 (Colab): 运行下面注释的代码上传")
    print("  from google.colab import files")
    print("  files.upload()  # 选择 aci_dqn_project.zip")
    print("  !unzip -q aci_dqn_project.zip -d /content/")
    print("")
    print("方式 2 (Kaggle): 在 Settings > Data > Upload 中上传项目 zip")
    print("")
    print("方式 3: git clone")
    print("  !git clone <repo_url> /content/ACI-DQN")
    print("=" * 60)
    raise FileNotFoundError(
        "Cannot find project root (src/utils.py + config.yaml). "
        "Upload project files first."
    )

print(f"项目根目录: {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
del _find_project_root

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch

# 确保项目根目录在 sys.path

from src.utils import load_config, ensure_dir, set_global_seed
from src.data_preprocess import run as preprocess_run
from src.datacenter_env import DataCenterEnv, action_to_n, n_to_action
from src.rl.dqn_agent import DQNAgent
from src.rl.train_dqn import (
    EpisodeStats,
    IdentityAugmenter,
    evaluate,
    train_agent,
    rollout_episode,
)
from src.rl.augmenters import (
    ConformalAugmenter,
    ForecastAugmenter,
    StaticConformalAugmenter,
)
from src.scenarios import (
    build_scenario_config,
    get_phase_config,
    apply_scenario_to_env,
    list_available_scenarios,
)
from _common import build_env_and_splits, calibration_arrival_data

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. 训练配置

修改下方参数选择要训练的方法、场景和强度。

In [ ]:
# ============================================================
# 配置区域 —— 按需修改
# ============================================================

CONFIG_PATH = "config.yaml"
SEED = 2024

# ---- 训练参数 ----
TRAIN_EPISODES = 500            # 训练 episode 数（调试时可用 2~10）
TRAINING_SEEDS = [2024]         # 训练种子列表，多 seed 会依次训练

# ---- 选择要训练的方法 ----
# 可用："dqn", "forecast_dqn", "static_conformal_dqn", "aci_dqn"
# 附录："shielded_dtaci_dqn"
METHODS_TO_TRAIN = ["dqn", "forecast_dqn", "static_conformal_dqn", "aci_dqn"]

# ---- 场景选择 ----
# 可选：E0_easy, E1_normal_hard, E2_distribution_shift,
#       E3_bursty_uncertainty, E4_capacity_cliff
# 设为 None 则使用 base config（无场景覆盖）
SCENARIO_ID = "E1_normal_hard"

# ---- 快速评估 ----
QUICK_EVAL_DAYS = 5             # 训练后快速评估的天数，设 0 跳过

# ============================================================
print(f"Methods: {METHODS_TO_TRAIN}")
print(f"Scenario: {SCENARIO_ID}")
print(f"Training episodes: {TRAIN_EPISODES}")
print(f"Training seeds: {TRAINING_SEEDS}")
print(f"Quick eval days: {QUICK_EVAL_DAYS}")

## 3. 加载配置与数据预处理

In [ ]:
# ---- 加载基础配置 ----
base_cfg = load_config(CONFIG_PATH)
set_global_seed(SEED)

# ---- 数据预处理（生成 processed_load.csv）----
print("=" * 60)
print("Stage 1: Data Preprocessing")
print("=" * 60)
preprocess_run(base_cfg)
print("Preprocessing done.")

# ---- 构建场景配置（如选择了场景）----
if SCENARIO_ID:
    scenario_cfg = build_scenario_config(SCENARIO_ID, base_cfg)
    train_cfg = get_phase_config(scenario_cfg, "train")
    cal_cfg = get_phase_config(scenario_cfg, "calibration")
    test_cfg = get_phase_config(scenario_cfg, "test")
    print(f"\nScenario '{SCENARIO_ID}' loaded:")
    print(f"  Nmax={train_cfg.get('server', {}).get('Nmax', base_cfg['server']['Nmax'])}")
    print(f"  ramp_limit={train_cfg.get('server', {}).get('ramp_limit', base_cfg['server']['ramp_limit'])}")
    print(f"  switch_cost={train_cfg.get('power', {}).get('switch_cost', base_cfg['power']['switch_cost'])}")
else:
    train_cfg = base_cfg
    cal_cfg = base_cfg
    test_cfg = base_cfg
    print("\nUsing base config (no scenario overlay)")

## 4. 构建环境与数据划分

In [ ]:
print("=" * 60)
print("Stage 2: Building Environment and Data Splits")
print("=" * 60)

# 构建基础环境（使用增强后的 train_cfg）
env, splits, norm_matrix = build_env_and_splits(train_cfg)

print(f"  Observation dim: {env.observation_dim}")
print(f"  Action dim (discrete bins): {env.action_dim}")
print(f"  Server range: [{env.Nmin}, {env.Nmax}]")
print(f"  Ramp limit: {env.cfg["server"]["ramp_limit"]}")
print(f"  Train days: {len(splits['train'])}")
print(f"  Calibration days: {len(splits['calibration'])}")
print(f"  Test days: {len(splits['test'])}")

## 5. 训练方法构建器

In [ ]:
def build_rl_agent(method: str, cfg: Dict, env: DataCenterEnv):
    """
    构建 DQNAgent 及其对应的 StateAugmenter。

    Returns
    -------
    (agent, augmenter) 元组
    """
    rl_cfg = cfg["rl"]
    K = cfg["qos"]["K"]

    if method == "dqn":
        aug = IdentityAugmenter()
        state_dim = env.observation_dim                     # 17

    elif method == "forecast_dqn":
        aug = ForecastAugmenter(cfg)
        state_dim = env.observation_dim + K                 # 17 + 3 = 20

    elif method == "static_conformal_dqn":
        aug = StaticConformalAugmenter(cfg)
        state_dim = env.observation_dim + 2 * K             # 17 + 6 = 23

    elif method == "aci_dqn":
        aug = ConformalAugmenter(cfg, learner="aci", use_shield=False)
        state_dim = env.observation_dim + 2 * K             # 17 + 6 = 23

    elif method == "shielded_dtaci_dqn":  # 附录
        aug = ConformalAugmenter(cfg, learner="dtaci", use_shield=True)
        state_dim = env.observation_dim + 2 * K             # 17 + 6 = 23

    else:
        raise ValueError(f"Unknown method: {method}. "
                         f"Available: dqn, forecast_dqn, static_conformal_dqn, "
                         f"aci_dqn, shielded_dtaci_dqn")

    agent = DQNAgent(
        state_dim=state_dim,
        action_dim=env.action_dim,
        hidden=rl_cfg["hidden_sizes"],
        lr=rl_cfg["lr"],
        gamma=rl_cfg["gamma"],
        batch_size=rl_cfg["batch_size"],
        replay_size=rl_cfg["replay_size"],
        epsilon_start=rl_cfg["epsilon_start"],
        epsilon_end=rl_cfg["epsilon_end"],
        epsilon_decay=rl_cfg["epsilon_decay"],
        target_update_interval=rl_cfg["target_update_interval"],
        learning_starts=rl_cfg["learning_starts"],
        reward_scale=rl_cfg["reward_scale"],
        max_grad_norm=rl_cfg["max_grad_norm"],
    )
    return agent, aug


def warm_up_augmenter(aug, cfg, norm_matrix, cal_indices):
    """
    对所有需要校准的 conformal augmenter 执行 warm-up。
    IdentityAugmenter 和 ForecastAugmenter 跳过。
    """
    method_name = getattr(aug, "name", type(aug).__name__)
    if method_name.lower() in ("identity", "forecastaugmenter"):
        return

    y_hat_cal, y_cal = calibration_arrival_data(
        cfg, norm_matrix, cal_indices
    )
    aug.warm_up_from_calibration(y_hat_cal, y_cal)
    print(f"  Warmed up on {len(cal_indices)} calibration days")

## 6. 训练循环

每个 episode：随机抽样训练日 → ε-greedy 探索 rollout → replay buffer 采样 → 梯度更新 → 衰减 ε。

In [ ]:
models_dir = ensure_dir(train_cfg["paths"]["models_dir"])
outputs_dir = ensure_dir(train_cfg["paths"]["outputs_dir"])
logs_dir = ensure_dir(train_cfg["paths"]["logs_dir"])

all_results = {}          # method -> list of (seed, history_df, model_path)
t_total_start = time.time()

for method in METHODS_TO_TRAIN:
    print("\n" + "=" * 60)
    print(f"Method: {method.upper()}")
    print("=" * 60)

    method_results = []

    for t_seed in TRAINING_SEEDS:
        print(f"\n  --- training_seed = {t_seed} ---")

        # 每个 seed 构建全新的 agent 和 augmenter
        agent, aug = build_rl_agent(method, train_cfg, env)
        print(f"  State dim: {agent.state_dim}, "
              f"Augmenter: {getattr(aug, 'name', type(aug).__name__)}")
        print(f"  Device: {agent.device}")

        # ---- 校准 warm-up（仅 conformal 方法需要）----
        warm_up_augmenter(aug, train_cfg, norm_matrix, splits["calibration"])

        # ---- 训练 ----
        rng = np.random.default_rng(t_seed + hash(method) % 1000)
        log_path = str(logs_dir / f"{method}_seed{t_seed}_train.log")

        t_start = time.time()
        history = train_agent(
            env, agent, aug, splits["train"],
            episodes=TRAIN_EPISODES,
            rng=rng,
            log_path=log_path,
        )
        elapsed = time.time() - t_start
        print(f"  Training done in {elapsed:.1f}s ({elapsed/60:.1f} min)")

        # ---- 保存模型 ----
        if len(TRAINING_SEEDS) > 1:
            model_name = f"{method}_seed{t_seed}"
        else:
            model_name = method
        model_path = models_dir / f"{model_name}.pt"
        agent.save(str(model_path))
        print(f"  Model saved -> {model_path}")

        # ---- 保存训练历史 ----
        history_df = pd.DataFrame(history)
        history_path = outputs_dir / f"{model_name}_training_history.csv"
        history_df.to_csv(history_path, index=False)

        method_results.append((t_seed, history_df, model_path))

    all_results[method] = method_results

total_elapsed = time.time() - t_total_start
print("\n" + "=" * 60)
print(f"All training done in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"  {total_elapsed/3600:.2f} hours")
print("=" * 60)

## 7. 快速评估（训练后）

在少量测试日上用 greedy 策略评估每个模型，输出成本和 SLA 统计。

In [ ]:
from src.evaluation.metrics import episode_stats_to_row

if QUICK_EVAL_DAYS > 0:
    print("=" * 60)
    print(f"Quick Evaluation ({QUICK_EVAL_DAYS} test days)")
    print("=" * 60)

    eval_days = splits["test"][:QUICK_EVAL_DAYS]

    # 构建评估环境（使用 test_cfg 覆盖）
    eval_env, _, _ = build_env_and_splits(test_cfg)
    apply_scenario_to_env(eval_env, test_cfg)

    eval_rows = []
    for method in METHODS_TO_TRAIN:
        for t_seed, _, model_path in all_results.get(method, []):
            agent, aug = build_rl_agent(method, test_cfg, eval_env)
            agent.load(str(model_path))
            agent.epsilon = 0.0

            # 校准 warm-up（conformal 方法使用 cal_cfg 数据）
            cal_env, _, cal_norm = build_env_and_splits(cal_cfg)
            warm_up_augmenter(aug, cal_cfg, cal_norm, splits["calibration"])

            eval_rng = np.random.default_rng(SEED + 30)
            stats_list = evaluate(eval_env, agent, aug, eval_days,
                                   rng=eval_rng, record_actions=False)
            for s in stats_list:
                row = episode_stats_to_row(method, s)
                row["training_seed"] = t_seed
                eval_rows.append(row)

    if eval_rows:
        eval_df = pd.DataFrame(eval_rows)
        agg_cols = [c for c in eval_df.columns
                    if c not in ("method", "day_index", "training_seed")]
        summary = eval_df.groupby("method")[agg_cols].mean()

        # 选择关键列展示
        show_cols = ["total_objective_cost", "electricity_cost", "qos_cost",
                     "average_active_servers", "average_utilization",
                     "P1_sla_violation_rate", "P2_sla_violation_rate",
                     "P3_sla_violation_rate"]
        show_cols = [c for c in show_cols if c in summary.columns]
        print("\n" + summary[show_cols].round(4).to_string())
else:
    print("Quick eval skipped (QUICK_EVAL_DAYS = 0)")

## 8. 下载产物

训练好的模型和训练历史在 `outputs/` 目录下，打包下载。

In [ ]:
import shutil

archive_name = "aci_dqn_outputs"
archive_path = shutil.make_archive(archive_name, "zip", "outputs")
print(f"Archive created: {archive_path}")
print(f"Size: {Path(archive_path).stat().st_size / 1024**2:.1f} MB")

# ---- Colab 下载（取消注释）----
# from google.colab import files
# files.download(archive_path)

# ---- Kaggle 下载（取消注释）----
# from IPython.display import FileLink
# display(FileLink(archive_path))

# ---- AutoDL / 通用：直接用 scp 或文件管理器下载 outputs 目录 ----
print("\nDownload the archive from the file panel, or use:")
print(f"  scp user@host:{archive_path} .")

## 9. 训练进度可视化（可选）

In [ ]:
import matplotlib.pyplot as plt

# 颜色映射
COLORS = {
    "dqn": "#2196F3",
    "forecast_dqn": "#3F51B5",
    "static_conformal_dqn": "#795548",
    "aci_dqn": "#9C27B0",
    "shielded_dtaci_dqn": "#E91E63",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
window = max(5, TRAIN_EPISODES // 50)  # 自适应平滑窗口

for method in METHODS_TO_TRAIN:
    color = COLORS.get(method, "#333333")
    for t_seed, history_df, _ in all_results.get(method, []):
        label = f"{method}" if len(TRAINING_SEEDS) == 1 else f"{method} (seed={t_seed})"
        rew = history_df["reward"].rolling(window, min_periods=1).mean()
        loss = history_df["avg_loss"].rolling(window, min_periods=1).mean()
        axes[0].plot(rew.values, color=color, alpha=0.8, linewidth=1.2, label=label)
        axes[1].plot(loss.values, color=color, alpha=0.8, linewidth=1.2, label=label)

axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Smoothed Reward")
axes[0].set_title(f"Training Reward (window={window})")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Smoothed TD Loss")
axes[1].set_title(f"Training Loss (window={window})")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/figures/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. 补充说明

### 切换为真实 Trace 数据

修改 `config.yaml`：
```yaml
workload:
  source: trace
  trace_dir: "trace_dataset/"
```
并将 trace CSV 放入对应目录。

### 本地完整评估

```bash
python main.py --scenario E1 --all --training-seeds 2024 2025 2026 --scenario-seeds 100 101 102
python generate_figures.py
```

### 单元测试

```bash
python -m pytest tests/ -v
```